<a href="https://colab.research.google.com/github/ankan-git-coder/Deep-learning-COURSE/blob/main/breast_cancer_classification_nn.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
import pandas as pd
import numpy as np
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

data_source = load_breast_cancer()
df = pd.DataFrame(data_source.data, columns=data_source.feature_names)
df['label'] = data_source.target

X = df.drop(columns='label', axis=1).values
Y = df['label'].values


X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.2, random_state=2)


scaler = StandardScaler()
X_train_std = scaler.fit_transform(X_train)
X_test_std = scaler.transform(X_test)


X_train_tensor = torch.FloatTensor(X_train_std)
Y_train_tensor = torch.LongTensor(Y_train)
X_test_tensor = torch.FloatTensor(X_test_std)
Y_test_tensor = torch.LongTensor(Y_test)


class BreastCancerModel(nn.Module):
    def __init__(self):
        super(BreastCancerModel, self).__init__()

        self.layer1 = nn.Linear(30, 20)

        self.layer2 = nn.Linear(20, 2)
        self.relu = nn.ReLU()

    def forward(self, x):
        x = self.relu(self.layer1(x))
        x = self.layer2(x)
        return x

model = BreastCancerModel()


criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.01)


epochs = 20
for epoch in range(epochs):
    optimizer.zero_grad()
    outputs = model(X_train_tensor)
    loss = criterion(outputs, Y_train_tensor)
    loss.backward()
    optimizer.step()

    if (epoch+1) % 5 == 0:
        print(f'Epoch [{epoch+1}/{epochs}], Loss: {loss.item():.4f}')


model.eval()
with torch.no_grad():
    test_outputs = model(X_test_tensor)
    _, predicted = torch.max(test_outputs, 1)
    accuracy = (predicted == Y_test_tensor).sum().item() / Y_test_tensor.size(0)
    print(f'\nAccuracy on Test Data: {accuracy * 100:.2f}%')


def predict_cancer(input_data):

    input_as_numpy = np.asarray(input_data).reshape(1, -1)

    input_std = scaler.transform(input_as_numpy)

    input_tensor = torch.FloatTensor(input_std)


    model.eval()
    with torch.no_grad():
        output = model(input_tensor)
        _, prediction = torch.max(output, 1)

    if prediction.item() == 0:
        return "The tumor is Malignant"
    else:
        return "The tumor is Benign"


sample_data = X_test[0]
result = predict_cancer(sample_data)
print(f"Prediction for sample: {result}")

Epoch [5/20], Loss: 0.3566
Epoch [10/20], Loss: 0.1694
Epoch [15/20], Loss: 0.1079
Epoch [20/20], Loss: 0.0805

Accuracy on Test Data: 95.61%
Prediction for sample: The tumor is Benign
